# 06 · An agent that learns from conversation

A support agent at Northwind handles customer escalations over chat. Everything
it knows about *how the team works* — who owns what, which policy applies to
whom — it learned because a human said it out loud. No system of record holds
any of it.

This notebook runs that agent for real: it assembles context, calls a model to
extract durable facts, and captures them back with provenance. Then the team
changes, the agent's memory contradicts itself, and we watch the governed loop
catch it — first deterministically, then with a model finding something
determinism cannot.

Every model call here goes to **Llama 3.3 70B Instruct** — open weights, via
OpenRouter. Add an `OPENROUTER_API_KEY` in Colab's Secrets panel (the key icon
in the left sidebar) and enable it for this notebook. **Without a key** every
cell still runs; the deterministic analyzers do all the detection. The floor is
always keyless.

## Setup

In [ ]:
# dejadb 1.0.5 is on PyPI; `dejadb.helpers` ships inside the wheel.
%pip install -q "dejadb>=1.0.5" matplotlib

from dejadb.helpers import *
import dejadb, json, os, urllib.request
print("dejadb", dejadb.__version__)

dejadb 1.0.5


## One small helper: talk to the model

One open-weights model, one key, one function — it powers both the agent's own
extraction step and Waiser's discovery pass later.

`chat()` returns `""` when no key is set, so nothing below breaks — it just
skips the model-dependent part.

In [ ]:
# OpenRouter only. Add OPENROUTER_API_KEY in Colab's Secrets panel
# (the key icon in the left sidebar) and switch it on for this notebook.
try:
    from google.colab import userdata
    os.environ.setdefault("OPENROUTER_API_KEY", userdata.get("OPENROUTER_API_KEY") or "")
except Exception:
    pass

KEY = os.environ.get("OPENROUTER_API_KEY", "")

# Llama 3.3 70B — open weights, and it runs every LLM step in this notebook:
# the agent's own extraction, and Waiser's discovery pass later.
OPEN_MODEL = "meta-llama/llama-3.3-70b-instruct"
MODEL = "openrouter:" + OPEN_MODEL if KEY else None

def chat(prompt, model=OPEN_MODEL, max_tokens=700):
    """One OpenRouter completion. Returns "" with no key, so every cell still runs."""
    if not KEY:
        return ""
    req = urllib.request.Request(
        "https://openrouter.ai/api/v1/chat/completions",
        json.dumps({"model": model, "max_tokens": max_tokens,
                    "messages": [{"role": "user", "content": prompt}]}).encode(),
        {"Authorization": "Bearer " + KEY, "Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=90) as r:
        return json.loads(r.read())["choices"][0]["message"]["content"]

print("LLM:", MODEL or "no key - deterministic analyzers only, which is always a safe floor")

LLM: openrouter:meta-llama/llama-3.3-70b-instruct


## The agent's turn loop

A real agent turn is four moves, and all four are here:

1. **assemble** — one CAL `ASSEMBLE` builds the context from what it already knows
2. **remember** — the raw turn is stored as an Observation
3. **extract** — the model proposes durable facts from the turn
4. **capture back** — each fact is written with `derived_from` pointing at the turn

Two details that matter more than they look:

**The instruction and the vocabulary are grains, not constants.** Both are
written to the store and read back through CAL on every turn. Nothing about how
this agent extracts lives in the code.

**The schema is pinned.** Extraction that invents its own subject/relation names
every turn is exactly why LLM-extraction memory layers can't see their own
contradictions — `billing_escalation.owner = John` and `billing.owned_by = Priya`
never collide, so nothing ever detects that they disagree. The vocabulary is
recalled from memory, then *enforced* in code rather than merely requested.

**Re-learning is a no-op.** If the agent hears something it already believes,
nothing is written. No duplicate bloat.

Watch the third turn: a customer asking where their order is teaches the agent
nothing durable, and it correctly stores no fact.

In [ ]:
db = fresh("northwind.db", ns="support", actor="agent:support-bot")

# The extraction instruction is memory, not code. Store it as a grain and the
# agent's behaviour becomes something you can recall, version, supersede and
# audit — instead of something you redeploy.
db.add_fact("prompt:memory_extractor", "template", """You are the memory layer of a support agent at Northwind.

Here is what you already know:
{context}

New conversation turn:
  {speaker}: {message}

Return ONLY a JSON array of durable team facts, using EXACTLY these
subject/relation pairs and no others:
{schema}

Each item: {{"subject": ..., "relation": ..., "object": ...}}
Only facts about how the team or its processes work — never the customer's
one-off question, never anything a system of record already holds (order
status, invoice totals). If the turn says nothing that fits the schema above,
return [].""")

# The controlled vocabulary is memory too. Extraction that invents its own
# subject/relation names each turn is precisely why LLM-extraction memory
# layers cannot see their own contradictions: "billing_escalation.owner = John"
# and "billing.owned_by = Priya" never collide. Pin the schema and they do.
db.add_fact("prompt:memory_extractor", "vocabulary", json.dumps({
    "billing_escalation": ["owner"],
    "refund_policy":      ["standard_window_days", "enterprise_window_days"],
}))

def from_memory(relation):
    """Read a prompt asset back out through CAL, like any other grain."""
    grains = json.loads(db.cal(
        f'RECALL facts WHERE subject = "prompt:memory_extractor" '
        f'AND relation = "{relation}"'))["grains"]
    return grains[0]["fields"]["object"]

TURNS = []          # (speaker, message, observation hash) — the episode log

def turn(speaker, message):
    """One agent turn: assemble context -> model extracts -> capture back."""
    template = from_memory("template")                       # the instruction
    SCHEMA   = json.loads(from_memory("vocabulary"))         # the vocabulary

    ctx = json.loads(db.cal(
        'ASSEMBLE "what I know" FROM team: (RECALL facts WHERE subject = "billing_escalation"), '
        '  policy: (RECALL facts WHERE subject = "refund_policy") '
        'BUDGET 400 tokens FORMAT markdown'))["text"] or "(nothing yet)"

    obs = json.loads(db.remember(message, observer=speaker))["observation"]
    TURNS.append((speaker, message, obs))

    schema = "\n".join(f"  {s}: {' | '.join(rs)}" for s, rs in SCHEMA.items())
    raw = chat(template.format(context=ctx, speaker=speaker, message=message, schema=schema))
    learned = []
    if raw:
        try:
            learned = json.loads(raw[raw.index("["):raw.rindex("]") + 1])
        except (ValueError, json.JSONDecodeError):
            learned = []
    # the schema is enforced here, not merely requested
    learned = [{**f, "object": str(f.get("object"))} for f in learned
               if f.get("relation") in SCHEMA.get(f.get("subject"), [])]
    # re-learning a value the agent already holds is a no-op, not a duplicate
    live = {(s, g["relation"], str(g["object"])) for s in SCHEMA for g in facts(db, s)}
    learned = [f for f in learned if (f["subject"], f["relation"], f["object"]) not in live]

    for f in learned:
        db.cal(f'ADD fact SET subject = "{f["subject"]}" SET relation = "{f["relation"]}" '
               f'SET object = "{f["object"]}" SET derived_from = "{obs}" '
               f'REASON "learned from a conversation turn"')
    return learned

# March: three turns of an onboarding conversation
for speaker, msg in [
    ("lead:dana",  "Quick onboarding note: escalate any billing issue to John, he owns that area."),
    ("lead:dana",  "Also — enterprise customers get a 45 day refund window, not the standard 30."),
    ("cust:maria", "Hi, where is my order #4471?"),      # not durable — should be ignored
]:
    print(f"{speaker}: {msg}")
    for f in turn(speaker, msg):
        print(f"   learned -> {f['subject']} {f['relation']} {f['object']}")
    print()

lead:dana: Quick onboarding note: escalate any billing issue to John, he owns that area.
   learned -> billing_escalation owner John

lead:dana: Also — enterprise customers get a 45 day refund window, not the standard 30.
   learned -> refund_policy enterprise_window_days 45
   learned -> refund_policy standard_window_days 30

cust:maria: Hi, where is my order #4471?


## Every fact traces back to the turn that taught it

`provenance()` is the credit-assignment query: given a conversation turn, what
did the agent come to believe because of it? This is what makes a bad session
precisely undoable later — and what an auditor actually wants.

In [ ]:
# Every fact traces back to the turn that taught it — the credit-assignment query.
for speaker, message, obs in TURNS:
    derived = json.loads(db.provenance(obs))
    print(f'{speaker}: "{message[:62]}…"' if len(message) > 62 else f'{speaker}: "{message}"')
    if not derived:
        print("   → nothing durable learned")
    for g in derived:
        print(f'   → {g["subject"]} · {g["relation"]} = {g["object"]}   [{g["hash"][:12]}…]')
    print()

lead:dana: "Quick onboarding note: escalate any billing issue to John, he …"
   → billing_escalation · owner = John   [89879062f755…]

lead:dana: "Also — enterprise customers get a 45 day refund window, not th…"
   → refund_policy · standard_window_days = 30   [f7f027f654d1…]
   → refund_policy · enterprise_window_days = 45   [1a39e696a057…]

cust:maria: "Hi, where is my order #4471?"
   → nothing durable learned


## July: the handover call

Nobody tells the agent to forget anything. It simply learns a new fact.

Now it holds **two live values** for `billing_escalation.owner`. A vector store
would return both, ranked by similarity, and let the model pick. Here, `owner` is
a *functional* relation — one live value by definition — so this is a structural
contradiction, not a ranking problem.

In [ ]:
# July: the handover call. Nobody tells the agent to forget anything.
print("lead:dana: Handover note — Priya owns billing now, John moved to platform.")
for f in turn("lead:dana", "Handover note — Priya owns billing now, John moved to platform."):
    print(f"   learned -> {f['subject']} {f['relation']} {f['object']}")

print("\nlive values for billing_escalation.owner:")
for f in facts(db, "billing_escalation", "owner"):
    print("  •", f["object"])

lead:dana: Handover note — Priya owns billing now, John moved to platform.
   learned -> billing_escalation owner Priya

live values for billing_escalation.owner:
  • Priya
  • John


## Waiser sees it — with zero model calls

No `model=` argument. No API key required. No tokens, no dollars. The analyzer
computes over typed grains and cites the evidence it used.

In [ ]:
db.waiser_run(full_sweep=True)                 # no model= : zero LLM calls
show_recs(db)

  [medium] [reversible ] "billing_escalation" holds 2 live values for functional relation "owner"


## Resolve it, with a reason

Applying the recommendation supersedes the stale value. `latest()` is the right
read for a functional relation — one current head — and `history()` shows the
chain that produced it.

Nothing was deleted. "Who did this agent think owned billing in June?" is still
an answerable question.

In [ ]:
rec = [r for r in recs(db) if "contradiction" in r["analyzer"]][0]
db.apply_recommendation(rec["hash"],
    because="confirmed on the handover call — Priya owns billing from July 1")

# `latest` is the right read for a functional relation: one current head.
print("who owns billing escalations now?")
print("  ", json.loads(db.latest("billing_escalation", "owner"))["fields"]["object"])

print("\nand the chain that got us there:")
for v in json.loads(db.history("billing_escalation", "owner")):
    print(f'  [{"superseded" if v["superseded_by"] else "LIVE      "}] {v["object"]}')

print("\nnothing was deleted — John is still answerable, just no longer current.")

who owns billing escalations now?
   Priya

and the chain that got us there:
  [LIVE      ] Priya
  [superseded] John

nothing was deleted — John is still answerable, just no longer current.


## Where the model earns its place

Now a case no deterministic analyzer can reach. Three facts, each individually
well-formed:

- the refund API enforces a 30-day window **for every tier**
- enterprise customers have a **45-day** refund window
- two enterprise refunds were rejected at 35 and 38 days

No single rule is violated. The *combination* is the problem, and seeing it
requires connecting three separate grains. That's what the LLM discovery pass is
for — and note the same 70B open-weights model does it. Proposer, grounder and
verifier *can* each be a different model; none of this needs a frontier one.

Any draft it produces must still survive **GROUND** (are the cited premises
really in those grains?) and **VERIFY** (an independent soundness check) before
it is even allowed to queue — and that filter is strict enough that a single
sweep can legitimately surface nothing. Sweeps are cheap and idempotent, which
is why Waiser is designed to run on a hook or a cron rather than once.

In [ ]:
# A support thread that no single analyzer can reason about:
# each of these facts is individually well-formed.
db.record_tool_call("refund_api", '{"error":"window_expired","days":35,"tier":"enterprise"}',
                    is_error=True, thread="esc-118")
db.record_tool_call("refund_api", '{"error":"window_expired","days":38,"tier":"enterprise"}',
                    is_error=True, thread="esc-119")
db.add_fact("refund_api", "enforces_window", "30 days for every tier")

print("deterministic pass first:")
db.waiser_run(full_sweep=True)
print("  ", len(recs(db)), "pending — nothing here connects the three facts\n")

# The same open model does discovery. Proposer, grounder and verifier *can*
# each be a different model — here one 70B open-weights model does all three.
DISCOVER = MODEL
if DISCOVER:
    print(f"LLM discovery pass ({DISCOVER}):")
    # Sweeps are cheap and idempotent — Waiser is built to run on a session-end
    # hook or a cron, not once. Drafts are filtered hard, so a single pass can
    # legitimately surface nothing; running again is normal operation.
    found = []
    for sweep in range(1, 4):
        db.waiser_run(full_sweep=True, model=DISCOVER)
        found = [r for r in recs(db) if r["analyzer"].startswith("waiser.llm")]
        if found:
            break
        print(f"   sweep {sweep}: no draft cleared GROUND -> VERIFY")
    for r in found:
        print("  ", r["summary"])
else:
    print("no key configured — deterministic analyzers only, which is always a safe floor")

deterministic pass first:
   1 pending — nothing here connects the three facts

LLM discovery pass (openrouter:meta-llama/llama-3.3-70b-instruct):
   sweep 1: no draft cleared GROUND -> VERIFY
   Potential inconsistency in refund policy


## Even the model's finding queues for review

It arrives carrying `origin = llm` and status `pending`. It **cannot**
auto-apply — that path is closed to LLM findings regardless of policy. A human
adopts it with a written reason, or dismisses it with one.

In [ ]:
llm_recs = [r for r in recs(db) if r["analyzer"].startswith("waiser.llm")]
for r in llm_recs:
    print("origin =", r.get("origin", "llm"), "| status =", r["status"])
    db.apply_recommendation(r["hash"],
        because="confirmed with revenue-ops — the enterprise SLA is the correct window")
    print("adopted, with a named reviewer and a written reason\n")
if not llm_recs:
    print("nothing from the model this run — the deterministic floor stands on its own")

origin = llm | status = pending
adopted, with a named reviewer and a written reason


## Even the instructions are memory

Nothing above hardcodes how the agent extracts. The prompt and the controlled
vocabulary are grains, read back through CAL on every turn — so changing the
agent's behaviour is a **supersession**, with the same review, the same audit
trail and the same rollback as any other change. No redeploy, and the previous
wording stays answerable.

In [ ]:
# The instruction is a grain, so changing how the agent extracts is a
# supersession — reviewable, reversible, and no redeploy.
live = json.loads(db.cal('RECALL facts WHERE subject = "prompt:memory_extractor" '
                         'AND relation = "template"'))["grains"][0]
db.supersede(live["hash"], "fact", json.dumps({
    "subject":  "prompt:memory_extractor",
    "relation": "template",
    "object":   live["fields"]["object"] +
                "\n\nNever infer a fact the speaker did not state outright."}))

for v in json.loads(db.history("prompt:memory_extractor", "template")):
    state = "superseded" if v["superseded_by"] else "LIVE      "
    print(f'[{state}] {len(v["object"]):4d} chars  ...{v["object"].strip().splitlines()[-1][:52]}')

[LIVE      ]  618 chars  ...Never infer a fact the speaker did not state outrigh
[superseded]  562 chars  ...return [].


## The whole episode, on the record

Every transition — proposed, applied, dismissed — is an immutable, hash-chained
grain carrying the actor and the reason. It travels with the memory file and it
is queryable.

---

### What this notebook showed

| | |
|---|---|
| An agent learning from conversation | a real turn loop: assemble → extract → capture back |
| Provenance | every fact traces to the turn that taught it |
| A contradiction the team created | two live values for one functional relation |
| Detection with **zero** model calls | deterministic, reproducible, free |
| Correction without erasure | supersede; the old belief stays answerable |
| The model finding what rules can't | a three-fact inconsistency, grounded and verified |
| Governance that holds either way | LLM findings queue; they never auto-apply |

Next: [`02_the_wrong_lesson.ipynb`](02_the_wrong_lesson.ipynb) — an approved
lesson that turns out to be wrong, measured regression, and an audited rollback.

In [ ]:
audit(db)

  [pending    ] Consolidate 2 exact-duplicate grains for "billing_escalation"
  [applied    ] Potential inconsistency in refund policy
  [applied    ] "billing_escalation" holds 2 live values for functional relation "


## The whole episode, on the record

Every transition — proposed, applied, dismissed — is an immutable, hash-chained
grain carrying the actor and the reason. It travels with the memory file and it
is queryable.

---

### What this notebook showed

| | |
|---|---|
| An agent learning from conversation | a real turn loop: assemble → extract → capture back |
| Provenance | every fact traces to the turn that taught it |
| A contradiction the team created | two live values for one functional relation |
| Detection with **zero** model calls | deterministic, reproducible, free |
| Correction without erasure | supersede; the old belief stays answerable |
| The model finding what rules can't | a three-fact inconsistency, grounded and verified |
| Governance that holds either way | LLM findings queue; they never auto-apply |

Next: [`02_the_wrong_lesson.ipynb`](02_the_wrong_lesson.ipynb) — an approved
lesson that turns out to be wrong, measured regression, and an audited rollback.